<a href="https://colab.research.google.com/github/mobadara/finbert-sentiment-analyzer-api/blob/main/notebooks/02_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


# 🚀 **Financial News Sentiment Analysis: Part 2 - Fine-Tuning & Custom Loss**

**Author:** [Muyiwa J. Obadara](https://portfolio-frontend-livid.vercel.app)

**Project:** [FinBERT Sentiment API Backend](https://github.com/mobadara/finbert-sentiment-analyzer-api/)

## **Objective**
This notebook consumes the cleaned dataset from our data engineering pipeline and fine-tunes the `ProsusAI/finbert` model.

During our Exploratory Data Analysis, we identified a severe class imbalance (61.6% Neutral). Because standard accuracy metrics are misleading on imbalanced datasets, we will evaluate the final model's performance using the **Macro F1-Score**.

## **Architectural Decision: Handling Imbalance**
Rather than utilizing traditional resampling libraries like `imblearn` (SMOTE or Random Oversampling), this pipeline handles imbalance natively at the algorithm level.

Generating synthetic data by interpolating contextualized word embeddings often produces invalid semantic representations, while naive oversampling causes large transformer models to overfit and memorize duplicated records. Instead, we will subclass the Hugging Face `Trainer` to apply **Class-Weighted Cross-Entropy Loss**. This computationally efficient approach preserves the integrity of our text data while mathematically forcing the gradient descent to penalize misclassifications of our minority classes (Positive/Negative) heavily.

In [29]:
import pandas as pd
import torch
import evaluate
import numpy as np
from datasets import Dataset, load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments
from IPython.display import display
from sklearn.utils.class_weight import compute_class_weight
from typing import Dict, Tuple, Union
from numpy.typing import ArrayLike
from transformers import Trainer
import torch.nn as nn


try:
    from google.colab import userdata
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
REPO_NAME = 'finbert-sentiment-analyzer-api'
GITHUB_USERNAME = 'mobadara'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## **Ingest the data**
The cleaned trained dataset is stored in the github repository and needs to be ingested into the runtime.

The original dataset already has a test split, and we will use that here.

In [2]:
train_dataset_path = f'https://raw.githubusercontent.com/{GITHUB_USERNAME}/{REPO_NAME}/main/datasets/cleaned_financial_phrasebank.csv'

df_train = pd.read_csv(train_dataset_path)
display(df_train.head())

,text,label_text,label
0,The Samsung Mobile Applications Store was laun...,neutral,1
1,"F-Secure , a developer of security solutions a...",neutral,1
2,The company serves customers in various indust...,neutral,1
3,The company reported net sales of 302 mln euro...,neutral,1
4,Microsoft last week also issued the first patc...,neutral,1


Now, at this point, we need to load our original test dataset from hugging face hub, and also set our cleaned trained dataset to the hugging face format.

In [3]:
test_dataset = load_dataset('FinanceMTEB/financial_phrasebank', split='test')
train_dataset = Dataset.from_pandas(df_train)

In [4]:
train_dataset

Dataset({
    features: ['text', 'label_text', 'label'],
    num_rows: 1263
})

In [5]:
test_dataset

Dataset({
    features: ['text', 'label_text', 'label'],
    num_rows: 1000
})

### **Setup**
Here, we define our model checkpoint for tokenization./

In [7]:
model_checkpoint = 'ProsusAI/finbert'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [11]:
def tokenize_function(batch: Dict) -> Dict:
    """
    Tokenizes a batch of text using the pre-trained tokenizer.

    This function takes a dictionary containing text data, tokenizes it,
    and applies padding and truncation as necessary.

    Args:
        batch (dict): A dictionary containing the text data, typically
                      with a key 'text' holding a list of strings to be tokenized.

    Returns:
        dict: A dictionary containing the tokenized input IDs, attention mask,
              and token type IDs, suitable for model input.
    """
    return tokenizer(batch['text'], padding='max_length', truncation=True)


In [12]:
tokenized_datasets = DatasetDict({
    'train': train_dataset.map(tokenize_function, batched=True),
    'test': test_dataset.map(tokenize_function, batched=True)
})

Map:   0%|          | 0/1263 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [13]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1263
    })
    test: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
})

In [14]:
columns_to_remove = ['text', 'label_text']
tokenized_datasets = tokenized_datasets.remove_columns(columns_to_remove)

Rename the `label` column to standard hugging face label name: `labels`.

In [15]:
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

We are using PyTorch, and we need to inform the dataset.

In [16]:
tokenized_datasets.set_format('torch')

In [17]:
print("Dataset cleaned and formatted successfully!")
print(tokenized_datasets)

Dataset cleaned and formatted successfully!
DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1263
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
})


### **Dynamic Class Weight Calculation**

To effectively address the class imbalance in our financial dataset, we compute **class weights** dynamically based on the training distribution.

We use the "balanced" heuristic from `scikit-learn`, which applies the following formula:
$$W_i = \frac{N}{C \times n_i}$$
Where $N$ is the total number of samples, $C$ is the total number of unique classes, and $n_i$ is the number of samples in class $i$.

By converting these calculated weights into a PyTorch tensor and assigning them to our active hardware device (GPU/CPU), we prepare them to be injected into our custom Hugging Face `Trainer`. This forces the Cross-Entropy Loss function to mathematically penalize the model more heavily for misclassifying the minority classes (Positive/Negative) compared to the dominant Neutral class.

In [19]:
# 1. Extract the numerical labels from the training DataFrame
train_labels = df_train["label"].values
classes = np.unique(train_labels)

# 2. Calculate the weights using the balanced formula
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_labels)

# 3. Convert weights to a PyTorch tensor and move to GPU if available
class_weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)

print(f"Hardware Device: {device}")
print(f"Calculated Class Weights: {class_weights_tensor}")

Hardware Device: cpu
Calculated Class Weights: tensor([2.5515, 0.5411, 1.3156])


### **Evaluation Metrics: Why Macro F1-Score?**

In highly imbalanced datasets, standard accuracy is a deceptive metric. To rigorously evaluate our model, we use the **Macro F1-Score**.

The F1-score is the harmonic mean of Precision and Recall. By specifying `average="macro"`, the metric calculates the F1-score for each class independently and then takes the unweighted average:
$$F1_{macro} = \frac{1}{C} \sum_{i=1}^{C} F1_i$$
This ensures that the model's performance on the rare Positive and Negative classes contributes to the final score exactly as much as the dominant Neutral class.

In [24]:
f1_metric = evaluate.load('f1')
acc_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred: Tuple[ArrayLike, ArrayLike]) -> Dict[str, float]:
    """
    Computes Macro F1-Score and Accuracy for model evaluation.

    This function takes the model's raw logits and actual labels, calculates
    predictions, and then computes the Macro F1-Score and standard Accuracy.

    Args:
        eval_pred (tuple[np.ndarray, np.ndarray]): A tuple containing two numpy arrays:
            - logits (np.ndarray): The raw prediction scores (logits) from the model.
            - labels (np.ndarray): The true labels for the evaluation set.

    Returns:
        dict[str, float]: A dictionary containing the computed metrics:
            - 'f1_macro' (float): The Macro F1-Score.
            - 'accuracy' (float): The standard Accuracy.
    """
    logits, labels = eval_pred

    # Convert raw logits into actual class predictions by finding the highest probability
    predictions = np.argmax(logits, axis=-1)

    # Calculate Macro F1
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]

    # Calculate Standard Accuracy (kept for reference, but F1 is our guiding star)
    acc = acc_metric.compute(predictions=predictions, references=labels)["accuracy"]

    return {"f1_macro": f1, "accuracy": acc}

print("Metrics function defined successfully!")

Metrics function defined successfully!


### **Custom Weighted Trainer**

By default, the Hugging Face `Trainer` uses standard Cross-Entropy Loss. We override the `compute_loss` method to inject our custom PyTorch `class_weights_tensor`. This directly penalizes the model during gradient descent when it misclassifies our minority classes.

In [28]:
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model: torch.nn.Module,
                     inputs: Dict[str, torch.Tensor],
                     return_outputs: bool = False,
                     **kwargs) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict]]:
        """
        Overrides the default loss calculation to apply class weights.

        This method is called by the Hugging Face Trainer during the training loop
        to calculate the loss. It incorporates `class_weights_tensor` into
        `CrossEntropyLoss` to handle class imbalance.

        Args:
            model (torch.nn.Module): The model being trained.
            inputs (Dict[str, torch.Tensor]): A dictionary of input tensors, typically
                                              including 'labels' and other model inputs.
            return_outputs (bool, optional): Whether to return model outputs along with the loss.
                                             Defaults to False.
            **kwargs: Additional keyword arguments passed to the method.

        Returns:
            Union[torch.Tensor, Tuple[torch.Tensor, Dict]]: The computed loss, or a tuple
            containing the loss and model outputs if `return_outputs` is True.
        """
        # 1. Pop the labels from the inputs dictionary
        labels = inputs.pop("labels")

        # 2. Pass the rest of the inputs through the model to get predictions (logits)
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # 3. Define the PyTorch loss function using our calculated weights
        # (Assuming class_weights_tensor is already defined and on the correct device)
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)

        # 4. Calculate the loss
        # We use .view(-1) to flatten the tensors, ensuring shapes match perfectly
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        # 5. Return loss (and outputs if requested by the Trainer)
        return (loss, outputs) if return_outputs else loss

print("Custom WeightedLossTrainer initialized!")

Custom WeightedLossTrainer initialized!


### **Model Initialization and Hyperparameters**

We initialize the `ProsusAI/finbert` model with a classification head configured for 3 labels (Positive, Negative, Neutral).

**Hyperparameter Strategy:**
* **Learning Rate (2e-5):** A conservatively small learning rate standard for fine-tuning Transformer models to avoid catastrophic forgetting of the pre-trained financial vocabulary.
* **Epochs (3):** Sufficient iterations for the model to adapt to our specific dataset constraints without aggressively overfitting.
* **Metric for Best Model:** We strictly track the `f1_macro` at the end of every epoch. The Trainer is configured to automatically reload the model weights that achieved the highest Macro F1-score upon completion.

In [30]:
print("Loading model architecture...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=3,
    ignore_mismatched_sizes=True # Ensures the classification head resets cleanly
)

Loading model architecture...


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
# 2. Define Hyperparameters via TrainingArguments
training_args = TrainingArguments(
    output_dir="./finbert-finetuned",
    eval_strategy="epoch",      # Evaluate against the test set every epoch
    save_strategy="epoch",            # Save a checkpoint every epoch
    learning_rate=2e-5,
    per_device_train_batch_size=16,   # Safe batch size for standard Google Colab T4 GPUs
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,                # L2 Regularization
    load_best_model_at_end=True,      # Crucial: restores the best weights based on our custom metric
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none"                  # Disables third-party logging (like WandB) for a clean output
)

In [33]:
# 3. Instantiate our Custom Weighted Trainer
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Model and Trainer successfully configured!")

Model and Trainer successfully configured!
